# Exploration

## Chargement et analyse rapide des données

Pour ce projet nous avons accès à un dossier contenant les csv "articles_metadata" et "clicks_sample" contenant respectivement des informations sur les articles et les clics (les clics fait pour accéder aux articles). Il contient aussi un dossier "click" contenant les csv des clics effectués chaque heures. Et enfin il contient un embedding déjà fait des articles au format pickle.

Commençons par importer et afficher les deux csv principaux.

In [ ]:
import polars as pl
from pathlib import Path

DATA_PATH = Path("../data/news-portal-user-interactions-by-globocom/")

artcl_df = pl.read_csv(DATA_PATH / "articles_metadata.csv")
clk_df = pl.read_csv(DATA_PATH / "clicks_sample.csv")

print(artcl_df.shape)
artcl_df.head()

Le csv des articles contient des informations sur la catégorie dans laquelle est l'article, la date de sa création, l'identifiant du publieur et le nombre de mots qu'il contient.

In [ ]:
print(clk_df.shape)
clk_df.head()

Le csv des clics contient plusieurs colonnes que je ne comprend pas complétement encore et sur lesquels je vais devoir chercher plus de détails.

Regardons maintenant un exemple d'un csv contenu dans le dossier clicks.

In [ ]:
clk_h0_df = pl.read_csv(DATA_PATH / "clicks/clicks_hour_000.csv")

print(clk_h0_df.shape)
clk_h0_df.head()

En regardant le premier csv contenu dans clicks, je comprend qu'il est enfaite le même que clicks_sample et que ce dernier est juste un exemple du dossier. Affichons le deuxième pour comparer.

In [ ]:
clk_h1_df = pl.read_csv(DATA_PATH / "clicks/clicks_hour_001.csv")

print(clk_h1_df.shape)
clk_h1_df.head()

Il sera plus facile de manipuler un grand csv plutôt que plusieurs petit, nous allons donc fusionner tous les csv de clics.

In [ ]:
clks_df = pl.scan_csv(DATA_PATH / "clicks").collect()

print(clks_df.shape)
clks_df.head()

On se retrouve donc avec un csv articles contenant 364 047 lignes et 5 colonnes et un csv clics contenant 2 988 181 lignes et 12 colonnes.

## Analyse poussée

### Format des données et compte

On remarque que le tableau clics contient que des str, alors que les données sont en int normalement. Tentons de les convertirs, si des valeurs sont inadéquates à ce type, nous aurons une erreur. Profitons en aussi pour convertir les données contenant des dates en format datetime et callons les sur le fuseau horaire du Brésil.

In [ ]:
clks_df = clks_df.cast(pl.Int64)

clks_df = clks_df.with_columns(
	pl.col(["session_start", "click_timestamp"])
	.cast(pl.Datetime("ms"))
	.dt.replace_time_zone("UTC")
	.dt.convert_time_zone("America/Sao_Paulo"))

clks_df.head()

Parfait ça à fonctionner donc toutes nos valeurs sont bien compatibles int. Maintenant passons à l'analyse, regardons d'abord si il y a des valeurs vides.

In [ ]:
print("Nombre de valeurs Null dans le tableau des cliques")
clks_df.null_count()

In [ ]:
print("Nombre de valeurs Null dans le tableau des articles")
artcl_df.null_count()

Parfait, il n'y a aucune valeurs manquante. Continuons avec le nombre d'utilisateurs et d'articles dans chacun des tableaux.

In [ ]:
print("Nombre d'utilisateur différent:", clks_df["user_id"].n_unique())
print("Nombre d'article différent dans le tableau clic:", clks_df["click_article_id"].n_unique())
print("Nombre d'article différent dans le tableau article:", artcl_df["article_id"].n_unique())

En regardant ces données on se rend compte qu'il y a à peu près 8 fois plus d'articles que d'article ayant été consultés. Il faudra donc faire attention de ne pas rentrer dans le piège du biais de popularité. C'est à dire que si nous donnons trop d'importance aux nombre de consultations des articles dans notre algorithme de recommendation, les même articles seront consultés en boucle, ne laissant pas leur chance aux articles manquant de popularités.

### Distribution

#### Tableau des clics

Regardons maintenant la distribution d'autres données pertinentes.

In [ ]:
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pio.templates["center_title"] = go.layout.Template(
	layout=go.Layout(title_x=0.45, title_font={"size":28})
)

pio.templates.default += "+center_title" #type:ignore


rows = 3
cols = 3

fig = make_subplots(rows=rows, cols=cols)

rows_itr = 0
cols_itr = 0
for col in ["click_environment", "click_deviceGroup", "click_os", "click_country", "click_region", "click_referrer_type", "session_start", "session_size", "click_timestamp"]:
	fig.add_trace(go.Histogram(x=clks_df[col], name="Variable: " + col), rows_itr+1, cols_itr+1)
	cols_itr += 1
	if cols_itr == cols:
		cols_itr = 0
		rows_itr += 1

fig.update_layout(title = "Distribution du tableau des clics")

fig.show()

Premièrement on se rend compte que les variables "click_environment" et "click_country" sont extrèmement concentré autour d'un seule valeur. Elle serons donc inutile à notre algo et nous pouvons donc les supprimer. Et c'est aussi le cas de "click_region". Pourquoi ? Car l'algorithme que nous souhaitons crée n'a pas pour cible spécialement le Brésil, donc utiliser des données concernant spécifiquement le Brésil ne sera pas efficace pour ce que nous souhaitons obtenir.

In [ ]:
clks_df = clks_df.drop("click_environment", "click_country", "click_region")

On se rend compte d'un deuxième problème en regardant les deux données de dates. On voit que les dernières ouvertures de sessions se terminent le 17 octobre, mais les derniers clics ont lieu le 13 novembre. C'est à dire qu'on a des sessions qui durent minimum 4 semaine ! C'est probablment dû, par exemple, a des utilisateurs qui ouvrent la page web sur leur téléphone, quitte l'application sans la fermé, et qui reviennent dessus des jours plus tard. Le problème avec ces données c'est que les session sont sensées nous aider à prédire des articles intéressant en fonction du cheminement de l'utlisateur lors de cette session. Mais avec des session aussi espacé, l'utlisateur ne pourrait potentiellement ne plus être interessé du tout par ce qu'il consultait au début de la session, et dans ce cas, on devrait juste l'orienté vers ses centres d'intérêt globaux.

Nous allons donc supprimer les sessions de plus de 4 heures, car il est peu probable qu'une session dure plus longtemps que ça et il faudra prévoir de mettre sois un temps maximum par session, sois une clôture automatique des sessions après une période d'inactivitée.

In [ ]:
import plotly.express as xp

print("Nombre de ligne avant la suppression : ", clks_df.shape[0])
clks_df = clks_df.filter((pl.col("click_timestamp") - pl.col("session_start")) <= pl.duration(hours=4))
print("Nombre de ligne après la suppression : ", clks_df.shape[0])
xp.histogram(clks_df["click_timestamp"], title="Distribution de 'click_timestamp'")

Il y a peu de lignes impactés, c'est une bonne nouvelle. On se rend compte maintenant que la distribution est plus sensées

#### Tableau des articles

In [ ]:
artcl_df.head()

Nous nous retrouvons dans le même cas avec une colonne formater en timestamp. Nous allons donc lui faire passer par le même formatage que précédement.

In [ ]:
artcl_df = artcl_df.with_columns(
	pl.col("created_at_ts")
	.cast(pl.Datetime("ms"))
	.dt.replace_time_zone("UTC")
	.dt.convert_time_zone("America/Sao_Paulo"))

artcl_df.head()

On remarque que certaines des dates de créations des articles sont arrivé des années avant nos première sessions. Visualisons tout ça ainsi que la distribution des autres variables.

In [ ]:
rows = 2
cols = 2

fig = make_subplots(rows=rows, cols=cols)

rows_itr = 0
cols_itr = 0
for col in ["category_id", "created_at_ts", "publisher_id", "words_count"]:
	fig.add_trace(go.Histogram(x=artcl_df[col], name="Variable: " + col), rows_itr+1, cols_itr+1)
	cols_itr += 1
	if cols_itr == cols:
		cols_itr = 0
		rows_itr += 1

fig.update_layout(title = "Distribution du tableau des articles")

fig.show()

Bon déjà on peut supprimer "publisher_id" car elle ne contient qu'une valeur, donc inutile.

In [ ]:
artcl_df.drop_in_place("publisher_id")

Ceci fait, on remarque un autre problème, la date de création des articles. Il y a des articles qui ont été créés des années avant la première session enregistré, et des articles créés après la denière session enregistré. On se rappelle que plus haut on a vu que pas tout les articles on été consultés, nous allons donc nous permettre de supprimer tout ceux concernées.

In [ ]:
print(f"Taille du catalogue original : {len(artcl_df)}")

artcl_df = artcl_df.filter(
	pl.col("article_id").is_in(clks_df["click_article_id"].unique().to_list())
)

print(f"Taille du catalogue après purge : {len(artcl_df)}")
xp.histogram(artcl_df["created_at_ts"], title="Distribution de 'created_at_ts'")

On supprime ainsi une bonne partie des articles, ce qui rendra plus rapide les algorithmes.

#### Compte des consultations

Pour finir notre analyse exploratoire, nous allons regarder l'activité des articles et des utilisateurs. C'est à dire que nous allons regarder combien d'articles chaque utilisateur a consulté, et combien de consultations ont reçu chaque article.

In [ ]:
clk_user_count = (
	clks_df.group_by("user_id").agg(pl.len().alias("click_count"))
	.group_by("click_count").agg(pl.len().alias("nb_users"))
)

xp.bar(clk_user_count, x="click_count", y="nb_users", log_x=True, title="Distribution des utilisateurs selon leur nombre de clics")

La première chose qu'on voit c'est qu'il y a 315 utilisateurs ayant un seul clic. Cela est du au fait qu'on a supprimé les clics des sessions trop longue, ce qui à donc créer des utilisateurs à un seul clics. Nous allons donc devoir supprimer toutes les sessions à un seul clic, mais aussi corrigé la taille des sessions. 

In [ ]:
clks_df = clks_df.with_columns(
	pl.len().over("session_id").alias("session_size")
)

clks_df.head()
print("Taille avant la supression des sessions à un clic:", clks_df.height)
clks_df = clks_df.remove(pl.col("session_size") < 2)
print("Taille après la supression des sessions à un clic:", clks_df.height)

Pour continuer notre analyse, on remarque que la plupart des utilisateurs on très peu de clics. Cela pourrait rendre la prédiction difficile vu que l'algorithme ne pourra pas déterminer la tendance de cette session. On voit aussi qu'il y a très peu de valeurs au dessus de 50, nous allons potentiellement les supprimer vu que ce sont des valeurs trop rares pour être utiles. Regardons plus en détail.

In [ ]:
clk_user_count.sort("click_count")

Nous allons les garder pour le moment et nous comparons peut être plus tard la différence de résultats si on les avaient supprimés.

Faisons maintenant la même chose mais avec le compte de consultation des articles.

In [ ]:
clk_artcl_count = clks_df.group_by("click_article_id").agg(pl.len().alias("click_count"))

binned_df = clk_artcl_count.with_columns(
	pl.col("click_count").cut(
		breaks=[10, 100, 1000, 10000],
		labels=["1-10", "11-100", "101-1k", "1k-10k", ">10k"]
	).alias("click_bucket")
)

distribution_log_df = (
	binned_df.group_by("click_bucket")
	.agg(pl.len().alias("nb_articles"))
	.sort("nb_articles", descending=True)
)

fig = xp.bar(distribution_log_df, x="click_bucket", y="nb_articles", text_auto=True, title="Distribution des article selon leur nombre de clics")
fig.show()

On remarque que la plupart des articles sont très peu consulté et seulement une infime partie d'entre eux dépasse les 1000 clics.

## Export

Nous allons maintenant enregistré notre tableau de données pour pouvoir l'utilisé dans d'autres notebooks.

In [24]:
clks_df.write_parquet("../dataframes/clks_df_cleaned.parquet")
artcl_df.write_parquet("../dataframes/artcl_df_cleaned.parquet")